Get User Data

In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv()

client_id = os.getenv("CLIENT_ID")

In [2]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [3]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [4]:
anime_data_client = AnimeDataClient(client_id)

In [5]:
anime_data = anime_data_client.get_cache()

Build features

In [ ]:
# from anime_features import AnimeFeatureBuilder

# builder = AnimeFeatureBuilder(
#     anime_data,
#     max_tfidf_features=3000,
#     n_svd_components=300
# )

# anime_df = builder.build_features()

# builder.svd_explained_variance

Convert each anime in df to vectors

In [ ]:
# recommender = SimilarityRecommender()
# anime_vectors = recommender.create_anime_vectors(anime_df)
# anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [9]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tune SVD components

In [ ]:
from anime_evaluation import HitRateEvaluator
from anime_features import AnimeFeatureBuilder

svd_component_results = []
n_runs = 100
max_features = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500]
components = [50, 100, 200, 300]
weights_uncertainty = [4.5]
tuning_top_ks = [5, 10]
for n_feature in max_features:
    for component in components:
        builder = AnimeFeatureBuilder(
            anime_data,
            max_tfidf_features=n_feature,
            n_svd_components=component,
        )

        component_anime_df = builder.build_features()

        recommender = SimilarityRecommender()
        recommender.create_anime_vectors(component_anime_df)
        component_anime_df_scaled = recommender.anime_df_scaled

        hitman = HitRateEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
        )

        (
            bayesian_results,
            bayesian_summary,
            best_bayesian_weights,
            baseline_results,
            baseline_summary,
        ) = hitman.tune_bayesian_uncertainty(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
        )


        average_metrics = bayesian_summary.merge(
            baseline_summary,
            on="k",
            how="left",
        ).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})
        average_metrics["component"] = component
        average_metrics["n_feature"] = n_feature
        average_metrics["svd_explained_variance"] = builder.svd_explained_variance

        svd_component_results.append(average_metrics)

svd_component_summary = (
    pd.concat(svd_component_results, ignore_index=True)
    .sort_values(["k", "avg_precision_at_k"], ascending=[True, False])
)

svd_component_summary

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
44,4.5,5,0.446,0.218082,0.071935,0.035174,2.23,0.098,0.125513,0.015806,0.020244,0.49,200,3000,0.315776
4,4.5,5,0.408,0.192632,0.065806,0.031070,2.04,0.098,0.125513,0.015806,0.020244,0.49,200,500,0.659109
46,4.5,5,0.408,0.212574,0.065806,0.034286,2.04,0.098,0.125513,0.015806,0.020244,0.49,300,3000,0.410419
52,4.5,5,0.406,0.189534,0.065484,0.030570,2.03,0.098,0.125513,0.015806,0.020244,0.49,200,3500,0.298553
10,4.5,5,0.398,0.193834,0.064194,0.031264,1.99,0.098,0.125513,0.015806,0.020244,0.49,100,1000,0.315495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29,4.5,10,0.256,0.124170,0.082581,0.040055,2.56,0.064,0.061167,0.020645,0.019731,0.64,200,2000,0.368236
1,4.5,10,0.243,0.112146,0.078387,0.036176,2.43,0.064,0.061167,0.020645,0.019731,0.64,50,500,0.280921
21,4.5,10,0.243,0.111242,0.078387,0.035884,2.43,0.064,0.061167,0.020645,0.019731,0.64,200,1500,0.412403
15,4.5,10,0.226,0.113369,0.072903,0.036571,2.26,0.064,0.061167,0.020645,0.019731,0.64,300,1000,0.616126


In [17]:
svd_component_summary[svd_component_summary['n_feature'] == 500]

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
4,4.5,5,0.408,0.192632,0.065806,0.031070,2.04,0.098,0.125513,0.015806,0.020244,0.49,200,500,0.659109
2,4.5,5,0.328,0.164581,0.052903,0.026545,1.64,0.098,0.125513,0.015806,0.020244,0.49,100,500,0.435009
6,4.5,5,0.282,0.177741,0.045484,0.028668,1.41,0.098,0.125513,0.015806,0.020244,0.49,300,500,0.817442
0,4.5,5,0.272,0.162107,0.043871,0.026146,1.36,0.098,0.125513,0.015806,0.020244,0.49,50,500,0.280921
5,4.5,10,0.273,0.126215,0.088065,0.040715,2.73,0.064,0.061167,0.020645,0.019731,0.64,200,500,0.659109
3,4.5,10,0.259,0.112002,0.083548,0.036130,2.59,0.064,0.061167,0.020645,0.019731,0.64,100,500,0.435009
1,4.5,10,0.243,0.112146,0.078387,0.036176,2.43,0.064,0.061167,0.020645,0.019731,0.64,50,500,0.280921
7,4.5,10,0.209,0.112002,0.067419,0.036130,2.09,0.064,0.061167,0.020645,0.019731,0.64,300,500,0.817442


In [ ]:
svd_component_summary

In [19]:
best_svd_components = (
    svd_component_summary
    .sort_values(["k", "avg_precision_at_k", "avg_hit_rate"], ascending=[True, False, False])
    .groupby("k")
    .head(1)
)

best_svd_components

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
44,4.5,5,0.446,0.218082,0.071935,0.035174,2.23,0.098,0.125513,0.015806,0.020244,0.49,200,3000,0.315776
45,4.5,10,0.340,0.142843,0.109677,0.046078,3.40,0.064,0.061167,0.020645,0.019731,0.64,200,3000,0.315776


## Results

This tuning run evaluated `max_tfidf_features` from **500** to **4500** and SVD components from **50** to **300**, using **100 runs**, uncertainty weight **4.5**, and `random_state=42` for reproducible holdout splits.

| k | Best max TF-IDF features | Best SVD components | Avg precision@k | Std precision@k | Avg hit rate | Avg hits | SVD explained variance |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 5 | 3000 | 200 | 0.446 | 0.218 | 0.0719 | 2.23 | 0.3158 |
| 10 | 3000 | 200 | 0.340 | 0.143 | 0.1097 | 3.40 | 0.3158 |

The same parameter set won for both `k=5` and `k=10`: **3000 TF-IDF features** with **200 SVD components**. It also beat the popularity baseline by a wide margin: baseline Precision@5 was **0.098** and baseline Precision@10 was **0.064**.

Nearby settings such as `3000` features with `300` components and `3500` features with `200` components were competitive, but the best setting is smaller than `300` components and gives the strongest top-k performance. The explained variance is not the deciding metric here; higher SVD variance did not consistently translate into better recommendation precision.

Conclusion: use **`max_tfidf_features=3000`** and **`n_svd_components=200`** as the default feature configuration.